<a href="https://colab.research.google.com/github/ireshvithanage/Statistical-Learning-e22410/blob/main/Assignment_6_Gaussian_Process_Regression_and_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment : Gaussian Process Regression and Linear Regression**
---
B. V. I. D. Vithanage - E/22/410

---

## **Gaussian Process Regression**


### Multivariate Gaussian Process Formulation for Heating and Cooling Loads

Since the Heating Load ($Y_1$) and Cooling Load ($Y_2$) are physically related thermal responses, it is advantageous to model them jointly using a **Multivariate Gaussian Process (MGP)** rather than treating them as two independent scalar processes. This approach allows the model to capture both the individual behavior of each load and the dependence between them.

For a building configuration $g$, the observed thermal load vector is defined as

$$
Y_g = X_g + \nu_g,
$$

where

- $X_g \in \mathbb{R}^2$ represents the latent (noise-free) heating and cooling loads,
- $\nu_g \sim \mathcal{N}(0,\Sigma_\nu)$ denotes a bivariate Gaussian noise term with covariance matrix $\Sigma_\nu$.

For a dataset containing $n$ building configurations, the stacked observation vector

$$
\mathscr{Y}_n =
\begin{bmatrix}
Y_{g_1} \\
Y_{g_2} \\
\vdots \\
Y_{g_n}
\end{bmatrix}
\in \mathbb{R}^{2n}
$$

follows the joint Gaussian distribution

$$
\mathscr{Y}_n
\sim
\mathcal{N}
\left(
\mu_{Y,n},
K_n + I_n \otimes \Sigma_\nu
\right),
$$

where $K_n$ is constructed using a matrix-valued kernel

$$
\kappa(g,g') \in \mathbb{R}^{2 \times 2}.
$$

The diagonal entries of each kernel block describe the covariance structure of the heating and cooling loads individually, while the off-diagonal entries capture the correlation between the two outputs.

For a new building configuration $g$, the posterior predictive mean of the latent thermal loads is given by

$$
\mathbb{E}[X_g \mid \mathscr{Y}_n = y_n]
=
\mu_g
+
K_{g,n}
\left(
K_n + I_n \otimes \Sigma_\nu
\right)^{-1}
\left(
y_n - \mu_{Y,n}
\right),
$$

where $K_{g,n}$ contains the cross-covariances between the new input and the observed configurations.

By jointly modeling Heating Load and Cooling Load, the multivariate GP exploits both the shared building characteristics ($X_1$–$X_8$) and the inherent thermodynamic relationship between the two energy demands. As a result, the model can produce more accurate predictions and better quantify uncertainty than separate univariate Gaussian Process models.

In [ ]:
import pandas as pnd
import kagglehub

import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel as C,
    WhiteKernel
)

from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load and clean dataset
dataset_dir = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
energy_data = pnd.read_csv(dataset_dir + "/ENB2012_data.csv").dropna()

# Input features and target variables
input_features = energy_data.iloc[:, :8]
target_outputs = energy_data[['Y1', 'Y2']]

# Split dataset
X_tr, X_te, y_tr, y_te = train_test_split(
    input_features,
    target_outputs,
    test_size=0.2,
    random_state=42
)

# Feature scaling
feature_scaler = StandardScaler()
X_tr_scaled = feature_scaler.fit_transform(X_tr)
X_te_scaled = feature_scaler.transform(X_te)

# Gaussian Process kernel
gp_kernel = (
    C(1.0, (1e-3, 1e3))
    * RBF(10, (1e-2, 1e2))
    + WhiteKernel(noise_level=1, noise_level_bounds=(1e-5, 1e1))
)

# Gaussian Process model
gpr_model = GaussianProcessRegressor(
    kernel=gp_kernel,
    n_restarts_optimizer=10,
    normalize_y=True
)

# Model training
gpr_model.fit(X_tr_scaled, y_tr)

# Predictions
predicted_loads, prediction_std = gpr_model.predict(
    X_te_scaled,
    return_std=True
)

# Performance metrics
print(f"Optimized Kernel: {gpr_model.kernel_}")
print(f"R² Score (Heating): {r2_score(y_te['Y1'], predicted_loads[:, 0]):.4f}")
print(f"R² Score (Cooling): {r2_score(y_te['Y2'], predicted_loads[:, 1]):.4f}")

# Visualization: Actual vs Predicted Heating Load
heating_plot = go.Figure()

# Reference line
heating_plot.add_trace(
    go.Scatter(
        x=[y_te['Y1'].min(), y_te['Y1'].max()],
        y=[y_te['Y1'].min(), y_te['Y1'].max()],
        mode='lines',
        line=dict(color='gray', dash='dash'),
        name='Perfect Prediction'
    )
)

# Predicted points
heating_plot.add_trace(
    go.Scatter(
        x=y_te['Y1'],
        y=predicted_loads[:, 0],
        mode='markers',
        marker=dict(color='cyan', size=6, opacity=0.7),
        name='Predicted Heating Load'
    )
)

heating_plot.update_layout(
    title='Gaussian Process Regression: Actual vs Predicted Heating Load',
    xaxis_title='Actual Heating Load (Y1)',
    yaxis_title='Predicted Heating Load',
    template='plotly_dark'
)

heating_plot.show()

Using Colab cache for faster access to the 'eergy-efficiency-dataset' dataset.
Optimized Kernel: 1.92**2 * RBF(length_scale=1.82) + WhiteKernel(noise_level=0.00524)
R² Score (Heating): 0.9973
R² Score (Cooling): 0.9816


## **Linear Regression**


### Multivariate Linear Regression for Predicting Energy Demand

To model the **predicted energy demand** ($Y$) as a function of multiple building environment parameters, we formulate the problem using **Multivariate Linear Regression**.

Instead of relying on a single predictor, each building configuration $i$ is represented by a feature vector $X_i$, which contains the selected influencing variables. The corresponding energy demand $Y_i$ is expressed as a linear combination of these features:

$$
Y_i = \beta X_i + \beta_0 + \nu_i,
$$

where:
- $\beta$ is the vector of regression coefficients,
- $\beta_0$ is the intercept term,
- $\nu_i \sim \mathcal{N}(0, \Sigma_\nu)$ represents Gaussian noise.

---

For a dataset containing $n$ observations, we introduce an augmented feature vector:

$$
\widetilde{x}_i = \begin{bmatrix} x_i \\ 1 \end{bmatrix}.
$$

By stacking all samples into a design matrix $\widetilde{X}$ and combining parameters into:

$$
W = \begin{bmatrix} \beta \\ \beta_0 \end{bmatrix},
$$

the model can be written in compact matrix form:

$$
Y = \widetilde{X} W + N.
$$

---

Assuming the noise is independent and identically distributed with covariance $\Sigma_\nu = \sigma^2 I$, the maximum likelihood estimation reduces to minimizing the squared error loss. Under the condition that $\widetilde{X}^T \widetilde{X}$ is invertible, the closed-form solution for the parameters is given by the **normal equation**:

$$
\widehat{W}_{\text{MLE}} =
(\widetilde{X}^T \widetilde{X})^{-1}
\widetilde{X}^T Y.
$$

---

By computing $\widehat{W}_{\text{MLE}}$, the model learns the optimal linear relationship between input features and energy demand. This provides a clear and interpretable mapping that quantifies how each building parameter contributes to the final predicted energy consumption.

In [ ]:
# =========================
# Data Loading
# =========================
dataset_path_lr = kagglehub.dataset_download(
    "programmer3/green-building-multi-source-environment-dataset"
)

data_frame = pnd.read_csv(dataset_path_lr + "/green_building_dataset.csv").dropna()

# =========================
# Feature Selection
# =========================
selected_inputs = [
    'ventilation_rate',
    'equipment_load',
    'heating_energy',
    'cooling_energy',
    'occupancy'
]

X_lr = data_frame[selected_inputs]
y_lr = data_frame['predicted_energy_demand']

# =========================
# Train-Test Split
# =========================
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr,
    y_lr,
    test_size=0.2,
    random_state=42
)

# =========================
# Model Training
# =========================
linear_model = LinearRegression()
linear_model.fit(X_train_lr, y_train_lr)

# =========================
# Prediction
# =========================
y_pred_lr = linear_model.predict(X_test_lr)

# =========================
# Evaluation
# =========================
print("Linear Regression Coefficients:\n")

for feature_name, coef_value in zip(selected_inputs, linear_model.coef_):
    print(f"{feature_name}: {coef_value:.4f}")

print(f"\nIntercept: {linear_model.intercept_:.4f}")

print(f"\nR2 Score: {r2_score(y_test_lr, y_pred_lr):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test_lr, y_pred_lr):.4f}")

# =========================
# Visualization
# =========================
fig_lr = go.Figure()

# Perfect prediction line (now red)
fig_lr.add_trace(
    go.Scatter(
        x=[y_test_lr.min(), y_test_lr.max()],
        y=[y_test_lr.min(), y_test_lr.max()],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Perfect Fit Line'
    )
)

# Predictions
fig_lr.add_trace(
    go.Scatter(
        x=y_test_lr,
        y=y_pred_lr,
        mode='markers',
        marker=dict(
            color='royalblue',
            size=7,
            opacity=0.75,
            line=dict(width=0.5, color='white')
        ),
        name='Linear Regression Predictions'
    )
)

fig_lr.update_layout(
    title='Linear Regression: Actual vs Predicted Energy Demand',
    xaxis_title='Actual Energy Demand',
    yaxis_title='Predicted Energy Demand',
    template='plotly_dark'
)

fig_lr.show()

Using Colab cache for faster access to the 'green-building-multi-source-environment-dataset' dataset.
Linear Regression Coefficients:

ventilation_rate: 0.0485
equipment_load: 0.0959
heating_energy: 0.2414
cooling_energy: 0.2520
occupancy: 0.0443

Intercept: 8.0652

R2 Score: 0.7760
Mean Squared Error: 20.9101
